In [3]:
library(nlme)
library(tidyverse)
library(lme4)
library(emmeans)
library(car)
library(multcomp)
library(MANOVA.RM)
library(dplyr)

# Install the package if needed (comment out after first run)
# install.packages("lmerTest")

# Load the package
library(lmerTest)

library(multcomp) # For glht (General Linear Hypotheses)
library(ggplot2)  # For plotting

── Attaching core tidyverse packages ──────────────────────── tidyverse 2.0.0 ──
✔ dplyr     1.1.4     ✔ readr     2.1.5
✔ forcats   1.0.0     ✔ stringr   1.5.1
✔ ggplot2   3.5.2     ✔ tibble    3.2.1
✔ lubridate 1.9.4     ✔ tidyr     1.3.1
✔ purrr     1.0.4     
── Conflicts ────────────────────────────────────────── tidyverse_conflicts() ──
✖ dplyr::collapse() masks nlme::collapse()
✖ dplyr::filter()   masks stats::filter()
✖ dplyr::lag()      masks stats::lag()
ℹ Use the conflicted package (<http://conflicted.r-lib.org/>) to force all conflicts to become errors
Loading required package: Matrix


Attaching package: ‘Matrix’


The following objects are masked from ‘package:tidyr’:

    expand, pack, unpack



Attaching package: ‘lme4’


The following object is masked from ‘package:nlme’:

    lmList


Welcome to emmeans.
Caution: You lose important information if you filter this package's results.
See '? untidy'

Loading required package: carData


Attaching package: ‘car’


The follo

In [ ]:
# install.packages('IRkernel')
# IRkernel::installspec()

Updating HTML index of packages in '.Library'

Making 'packages.html' ...
 done



In [6]:
# creates a new data frame from an existing one containing repeated
# measures as columns, e.g. a data frame named exp1.df:
# subject	age	repeat1	repeat2
# 001		34	1.45	1.67
# 002		38	1.20	1.54
# ...
# called like:
# make.rm(1:2,3:4,exp1.df)
# would multiply the "constant" variables by the number of
# repeats and reformat the repeats to a single column.
# subject	age	repdat	contrasts
# 001		34	1.45	T1
# 002		38	1.20	T1
# ...
# 001		34	1.67	T2
# 002		38	1.54	T2
# ...
# this allows a "univariate" repeated measures analysis of the data.

make.rm <- function(constant, repeated, data, contrasts) {
  if(!missing(constant) && is.vector(constant)) {
    if(!missing(repeated) && is.vector(repeated)) {
      if(!missing(data)) {
        dd <- dim(data)
        replen <- length(repeated)
        if(missing(contrasts))
          contrasts <- ordered(sapply(paste("T", 1:length(repeated), sep=""), rep, dd[1]))
        else
          contrasts <- matrix(sapply(contrasts, rep, dd[1]), ncol=dim(contrasts)[2])
        if(length(constant) == 1) cons.col <- rep(data[,constant], replen)
        else cons.col <- lapply(data[,constant], rep, replen)
        new.df <- data.frame(cons.col,
                            repdat=as.vector(data.matrix(data[,repeated])),
                            contrasts)
        return(new.df)
      }
    }
  }
  cat("Usage: make.rm(constant, repeated, data [, contrasts])\n")
  cat("\tWhere 'constant' is a vector of indices of non-repeated data and\n")
  cat("\t'repeated' is a vector of indices of the repeated measures data.\n")
}

In [1]:
file_name <- '/Users/samgritz/Desktop/Rutgers/Milstein_Lab/Code/Rutgers-Neuroscience-PhD/GNB1_Paper_Analysis/paper_data/E_I_data/E_I_EPSP_amplitudes_R_format.csv'
E_I_experiment <- read.csv(file_name)
cat("✓ Loaded data from:", file_name, "\n")
cat("  Rows:", nrow(E_I_experiment), "\n")
cat("  Columns:", paste(colnames(E_I_experiment), collapse=", "), "\n\n")


✓ Loaded data from: /Users/samgritz/Desktop/Rutgers/Milstein_Lab/Code/Rutgers-Neuroscience-PhD/GNB1_Paper_Analysis/paper_data/E_I_data/E_I_EPSP_amplitudes_R_format.csv 
  Rows: 365 
  Columns: Subject, Drug, Pathway, Genotype, ISI10, ISI25, ISI50, ISI100, ISI300 



In [2]:
ISI_cols <- c("ISI10", "ISI25", "ISI50", "ISI100", "ISI300")
E_I_experiment_clean <- E_I_experiment %>%
  filter(complete.cases(!!!syms(ISI_cols)))

cat("  Filtered to complete cases:", nrow(E_I_experiment_clean), "rows\n")




ERROR: Error in E_I_experiment %>% filter(complete.cases(!!!syms(ISI_cols))): could not find function "%>%"


In [3]:
# Convert to long format
E_I_experiment_long <- E_I_experiment_clean %>%
  pivot_longer(
    cols = all_of(ISI_cols),
    names_to = "ISI_Time",
    values_to = "EPSP_Amplitude"
  ) %>%
  mutate(
    # Ensure all categorical variables are factors
    Genotype = as.factor(Genotype),
    Drug = as.factor(Drug),
    Pathway = as.factor(Pathway),
    Subject = as.factor(Subject),
    ISI_Time = factor(ISI_Time, levels = ISI_cols)
  )

cat("  Converted to long format:", nrow(E_I_experiment_long), "rows\n")
cat("  Subjects (cells):", length(unique(E_I_experiment_long$Subject)), "\n")
cat("  Genotypes:", paste(levels(E_I_experiment_long$Genotype), collapse=", "), "\n")
cat("  Drugs (0=Control, 1=Gabazine):", paste(levels(E_I_experiment_long$Drug), collapse=", "), "\n")
cat("  Pathways (1=Perforant, 2=Schaffer, 3=Basal):", paste(levels(E_I_experiment_long$Pathway), collapse=", "), "\n\n")

ERROR: Error in E_I_experiment_clean %>% pivot_longer(cols = all_of(ISI_cols), : could not find function "%>%"


In [4]:
pathway_codes <- list(
  "1" = "Perforant",
  "2" = "Schaffer",
  "3" = "Basal_Stratum_Oriens"
)

In [ ]:
# ******* posthoc only if genotype * ISI_time (interaction) is sig

#EPSP AMP:
#Print out the p-values from emmeans without adjustment into a table
#Then correct them with FDR like this:

#FDR correct these three groups seperately

#Corect these three - 15 total p-values
# gnb1 pp (con vs gabazine)

#these three = 15 total p-values
# wt vs gnb1 (sc, gabazine)

#These three 15 total 
# gnb1 so (con vs gabazine)

#For Supralinearity (Gabazine - per pathway) (5 corrections for each pathway)

#, E:I (Per pathway)- 5 corrections for each pathway

#posthoc only if genotype * ISI_time (interaction) is sig

In [12]:
# #Model 1: Main Effect of Genotype - offset of across data
# model_1 <- tryCatch({
# lmer(
#     EPSP_Amplitude ~ Genotype + (1 | Subject),
#     data = subset_data,
#     REML = TRUE
# )
# }, error = function(e) {
# cat("  Error fitting model:", conditionMessage(e), "\n")
# return(NULL)
# })

# #Model 2: Interaction or Slopes of data between genotype
# model_2 <- tryCatch({
# lmer(
#     EPSP_Amplitude ~ Genotype * ISI_Time + (1 | Subject),
#     data = subset_data,
#     REML = TRUE)})

### Perforant Data

In [13]:
# Filter: Perforant Path (1) + Gabazine (1)
pp_gabazine_data <- E_I_experiment_long %>% 
  filter(Pathway == 1 & Drug == 1)

In [14]:
# Fit models
cat("------- COMPARISON 1: WT vs GNB1 - Gabazine -------\n\n")

model_1 <- lmer(
  EPSP_Amplitude ~ Genotype + Genotype * ISI_Time + (1 | Subject),
  data = pp_gabazine_data,
  REML = TRUE
)

print(anova(model_1))

if (anova(model_1)$`Pr(>F)`[3] < 0.05) {
  emm_1 <- emmeans(model_1, pairwise ~ Genotype | ISI_Time)
  contrasts_1 <- summary(emm_1$contrasts, adjust = "none")
  cat("Genotype Contrasts (WT vs GNB1 at each ISI - UNCORRECTED):\n")
  print(contrasts_1)
  cat("\n")
}



------- COMPARISON 1: WT vs GNB1 - Gabazine -------

Type III Analysis of Variance Table with Satterthwaite's method
                   Sum Sq Mean Sq NumDF DenDF F value    Pr(>F)    
Genotype            23.39   23.39     1    23  2.3449 0.1393342    
ISI_Time          2097.63  524.41     4    92 52.5664 < 2.2e-16 ***
Genotype:ISI_Time  228.18   57.05     4    92  5.7183 0.0003725 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1
Genotype Contrasts (WT vs GNB1 at each ISI - UNCORRECTED):
ISI_Time = ISI10:
 contrast  estimate   SE   df t.ratio p.value
 GNB1 - WT    7.430 2.19 41.8   3.395  0.0015

ISI_Time = ISI25:
 contrast  estimate   SE   df t.ratio p.value
 GNB1 - WT    4.037 2.19 41.8   1.844  0.0722

ISI_Time = ISI50:
 contrast  estimate   SE   df t.ratio p.value
 GNB1 - WT    2.483 2.19 41.8   1.135  0.2630

ISI_Time = ISI100:
 contrast  estimate   SE   df t.ratio p.value
 GNB1 - WT    0.564 2.19 41.8   0.257  0.7981

ISI_Time = ISI300:
 contrast  estimate  

In [15]:
cat("------- COMPARISON 2: WT - Control vs Gabazine -------\n\n")

# Filter: Perforant Path (1) + WT
pp_wt_data <- E_I_experiment_long %>% 
  filter(Pathway == 1 & Genotype == "WT")

# Fit model
model_2 <- lmer(
  EPSP_Amplitude ~ Drug + Drug * ISI_Time + (1 | Subject),
  data = pp_wt_data,
  REML = TRUE
)

print(anova(model_2))

if (anova(model_2)$`Pr(>F)`[3] < 0.05) {
  emm_2 <- emmeans(model_2, pairwise ~ Drug | ISI_Time)
  contrasts_2 <- summary(emm_2$contrasts, adjust = "none")
  cat("Drug Contrasts (Control vs Gabazine at each ISI - UNCORRECTED):\n")
  print(contrasts_2)
  cat("\n")}





------- COMPARISON 2: WT - Control vs Gabazine -------

Type III Analysis of Variance Table with Satterthwaite's method
              Sum Sq Mean Sq NumDF  DenDF F value    Pr(>F)    
Drug          697.80  697.80     1 134.81  90.783 < 2.2e-16 ***
ISI_Time      564.84  141.21     4 126.69  18.371 6.312e-12 ***
Drug:ISI_Time 134.05   33.51     4 126.69   4.360  0.002453 ** 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1
Drug Contrasts (Control vs Gabazine at each ISI - UNCORRECTED):
ISI_Time = ISI10:
 contrast      estimate   SE  df t.ratio p.value
 Drug0 - Drug1    -6.68 1.02 129  -6.516  <.0001

ISI_Time = ISI25:
 contrast      estimate   SE  df t.ratio p.value
 Drug0 - Drug1    -7.24 1.02 129  -7.064  <.0001

ISI_Time = ISI50:
 contrast      estimate   SE  df t.ratio p.value
 Drug0 - Drug1    -3.96 1.02 129  -3.868  0.0002

ISI_Time = ISI100:
 contrast      estimate   SE  df t.ratio p.value
 Drug0 - Drug1    -2.69 1.02 129  -2.622  0.0098

ISI_Time = ISI300:
 cont

In [16]:
# ==============================================================================
# COMPARISON 3: GNB1 - Control vs Gabazine (Perforant Path)
# ==============================================================================
cat("------- COMPARISON 3: GNB1 - Control vs Gabazine -------\n\n")

# Filter: Perforant Path (1) + GNB1
pp_gnb1_data <- E_I_experiment_long %>% 
  filter(Pathway == 1 & Genotype == "GNB1")

# Fit model
model_3 <- lmer(
  EPSP_Amplitude ~ Drug + Drug * ISI_Time + (1 | Subject),
  data = pp_gnb1_data,
  REML = TRUE
)

print(anova(model_3))

if (anova(model_3)$`Pr(>F)`[3] < 0.05) {
  emm_3 <- emmeans(model_3, pairwise ~ Drug | ISI_Time)
  contrasts_3 <- summary(emm_3$contrasts, adjust = "none")
  print(contrasts_3)
}

------- COMPARISON 3: GNB1 - Control vs Gabazine -------

Type III Analysis of Variance Table with Satterthwaite's method
               Sum Sq Mean Sq NumDF  DenDF F value    Pr(>F)    
Drug          1393.20 1393.20     1 111.96 162.917 < 2.2e-16 ***
ISI_Time      1448.49  362.12     4 105.18  42.346 < 2.2e-16 ***
Drug:ISI_Time  547.77  136.94     4 105.18  16.014 2.868e-10 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1
ISI_Time = ISI10:
 contrast      estimate   SE  df t.ratio p.value
 Drug0 - Drug1   -13.35 1.19 107 -11.237  <.0001

ISI_Time = ISI25:
 contrast      estimate   SE  df t.ratio p.value
 Drug0 - Drug1   -10.71 1.19 107  -9.015  <.0001

ISI_Time = ISI50:
 contrast      estimate   SE  df t.ratio p.value
 Drug0 - Drug1    -6.63 1.19 107  -5.583  <.0001

ISI_Time = ISI100:
 contrast      estimate   SE  df t.ratio p.value
 Drug0 - Drug1    -3.30 1.19 107  -2.782  0.0064

ISI_Time = ISI300:
 contrast      estimate   SE  df t.ratio p.value
 Drug0 - Drug1

In [17]:
# ==============================================================================
# SUMMARY TABLE: All Uncorrected P-values
# ==============================================================================
cat("==============================================================================\n")
cat("SUMMARY TABLE: All Uncorrected P-values for Perforant Path\n")
cat("==============================================================================\n\n")

# Create summary dataframe
summary_df <- data.frame(
  Comparison = c(
    rep("1_WT_vs_GNB1_Gabazine", 5),
    rep("2_WT_Control_vs_Gabazine", 5),
    rep("3_GNB1_Control_vs_Gabazine", 5)
  ),
  ISI = rep(c("ISI10", "ISI25", "ISI50", "ISI100", "ISI300"), 3),
  Estimate = c(
    contrasts_1$estimate,
    contrasts_2$estimate,
    contrasts_3$estimate
  ),
  SE = c(
    contrasts_1$SE,
    contrasts_2$SE,
    contrasts_3$SE
  ),
  df = c(
    contrasts_1$df,
    contrasts_2$df,
    contrasts_3$df
  ),
  t_ratio = c(
    contrasts_1$t.ratio,
    contrasts_2$t.ratio,
    contrasts_3$t.ratio
  ),
  p_value_uncorrected = c(
    contrasts_1$p.value,
    contrasts_2$p.value,
    contrasts_3$p.value
  )
)

print(summary_df)
cat("\n")


SUMMARY TABLE: All Uncorrected P-values for Perforant Path

                   Comparison    ISI    Estimate       SE        df
1       1_WT_vs_GNB1_Gabazine  ISI10   7.4299666 2.188821  41.77564
2       1_WT_vs_GNB1_Gabazine  ISI25   4.0366757 2.188821  41.77564
3       1_WT_vs_GNB1_Gabazine  ISI50   2.4834200 2.188821  41.77564
4       1_WT_vs_GNB1_Gabazine ISI100   0.5635333 2.188821  41.77564
5       1_WT_vs_GNB1_Gabazine ISI300  -0.1991013 2.188821  41.77564
6    2_WT_Control_vs_Gabazine  ISI10  -6.6772943 1.024694 129.04189
7    2_WT_Control_vs_Gabazine  ISI25  -7.2385648 1.024694 129.04189
8    2_WT_Control_vs_Gabazine  ISI50  -3.9638698 1.024694 129.04189
9    2_WT_Control_vs_Gabazine ISI100  -2.6869317 1.024694 129.04189
10   2_WT_Control_vs_Gabazine ISI300  -3.1128921 1.024694 129.04189
11 3_GNB1_Control_vs_Gabazine  ISI10 -13.3459877 1.187729 106.74842
12 3_GNB1_Control_vs_Gabazine  ISI25 -10.7072333 1.187729 106.74842
13 3_GNB1_Control_vs_Gabazine  ISI50  -6.6312724 1.18772

### Schaffer Pathway

In [18]:
sc_gabazine_data <- E_I_experiment_long %>% 
  filter(Pathway == 2)

model_5 <- lmer(
  EPSP_Amplitude ~ Genotype + Genotype * ISI_Time + (1 | Subject),
  data = sc_gabazine_data,
  REML = TRUE
)

anova(model_5)

#if Genotype:ISI_Time interaction is significant then do post-hoc tests, if not, else do not do post-hoc tests
if (anova(model_5)$`Pr(>F)`[3] < 0.05) {
  emm_5 <- emmeans(model_5, pairwise ~ Genotype | ISI_Time)
  contrasts_5 <- summary(emm_5$contrasts, adjust = "none")
  print(contrasts_5)
} else {
  cat("Genotype:ISI_Time interaction is not significant. No post-hoc tests.")
}







,Sum Sq,Mean Sq,NumDF,DenDF,F value,Pr(>F)
,<dbl>,<dbl>,<int>,<dbl>,<dbl>,<dbl>
Genotype,0.1744628,0.1744628,1,30.60178,0.03275807,8.575673e-01
ISI_Time,1807.9603812,451.9900953,4,233.65621,84.86806674,2.095941e-44
Genotype:ISI_Time,16.6982620,4.1745655,4,233.65621,0.78383864,5.366722e-01


Genotype:ISI_Time interaction is not significant. No post-hoc tests.

In [19]:
cat("------- COMPARISON 2 Schaffer: WT - Control vs Gabazine -------\n\n")

sc_wt_data <- E_I_experiment_long %>% 
  filter(Pathway == 2 & Genotype == "WT")

model_6 <- lmer(EPSP_Amplitude ~ Drug + Drug * ISI_Time + (1 | Subject), data = sc_wt_data, REML = TRUE)

print(anova(model_6))

if (anova(model_6)$`Pr(>F)`[3] < 0.05) {
  emm_6 <- emmeans(model_6, pairwise ~ Drug | ISI_Time)
  contrasts_6 <- summary(emm_6$contrasts, adjust = "none")
  print(contrasts_6)
  } else {
      cat("Genotype:ISI_Time interaction not significant\n")
  }
      


------- COMPARISON 2 Schaffer: WT - Control vs Gabazine -------

Type III Analysis of Variance Table with Satterthwaite's method
              Sum Sq Mean Sq NumDF  DenDF F value    Pr(>F)    
Drug          226.30 226.299     1 128.14 72.9852 3.296e-14 ***
ISI_Time      900.61 225.152     4 122.71 72.6153 < 2.2e-16 ***
Drug:ISI_Time  45.14  11.284     4 122.71  3.6393  0.007732 ** 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1
ISI_Time = ISI10:
 contrast      estimate    SE  df t.ratio p.value
 Drug0 - Drug1    -3.12 0.659 124  -4.733  <.0001

ISI_Time = ISI25:
 contrast      estimate    SE  df t.ratio p.value
 Drug0 - Drug1    -4.24 0.659 124  -6.443  <.0001

ISI_Time = ISI50:
 contrast      estimate    SE  df t.ratio p.value
 Drug0 - Drug1    -3.22 0.659 124  -4.889  <.0001

ISI_Time = ISI100:
 contrast      estimate    SE  df t.ratio p.value
 Drug0 - Drug1    -1.89 0.659 124  -2.872  0.0048

ISI_Time = ISI300:
 contrast      estimate    SE  df t.ratio p.value
 D

In [20]:
cat("------- COMPARISON 3 Schaffer: GNB1 - Control vs Gabazine -------\n\n")

sc_gnb1_data <- E_I_experiment_long %>% 
  filter(Pathway == 2 & Genotype == "GNB1")

model_7 <- lmer(
  EPSP_Amplitude ~ Drug + Drug * ISI_Time + (1 | Subject),
  data = sc_gnb1_data,
  REML = TRUE
)

print(anova(model_7))

#if Genotype:ISI_Time interaction is significant then do post-hoc tests, if not, then dont
if (anova(model_7)$`Pr(>F)`[3] < 0.05) {
  emm_7 <- emmeans(model_7, pairwise ~ Drug | ISI_Time)
  contrasts_7 <- summary(emm_7$contrasts, adjust = "none")
  print(contrasts_5)
}



------- COMPARISON 3 Schaffer: GNB1 - Control vs Gabazine -------

Type III Analysis of Variance Table with Satterthwaite's method
              Sum Sq Mean Sq NumDF  DenDF F value    Pr(>F)    
Drug          212.84 212.843     1 102.91 67.7399 6.127e-13 ***
ISI_Time      998.33 249.582     4 101.00 79.4326 < 2.2e-16 ***
Drug:ISI_Time  64.61  16.152     4 101.00  5.1405 0.0008234 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


ERROR: Error in h(simpleError(msg, call)): error in evaluating the argument 'x' in selecting a method for function 'print': object 'contrasts_5' not found


In [21]:
E_I_experiment_long

Subject,Drug,Pathway,Genotype,ISI_Time,EPSP_Amplitude
<fct>,<fct>,<fct>,<fct>,<fct>,<dbl>
20240125_c5,0,1,WT,ISI10,4.537582
20240125_c5,0,1,WT,ISI25,4.141777
20240125_c5,0,1,WT,ISI50,3.773643
20240125_c5,0,1,WT,ISI100,2.252488
20240125_c5,0,1,WT,ISI300,2.011816
20240129_c1,0,1,GNB1,ISI10,16.642797
20240129_c1,0,1,GNB1,ISI25,14.807366
20240129_c1,0,1,GNB1,ISI50,9.594489
20240129_c1,0,1,GNB1,ISI100,8.671199


### Stratum Oriens pathway

In [22]:
so_gabazine_data <- E_I_experiment_long %>% 
  filter(Pathway == 3 & Drug == 1)

In [23]:
model_8 <- lmer(
  EPSP_Amplitude ~ Genotype + Genotype * ISI_Time + (1 | Subject),
  data = so_gabazine_data,
  REML = TRUE
)

print(anova(model_8))

#if Genotype:ISI_Time interaction is significant then do post-hoc tests, if not, then dont
if (anova(model_8)$`Pr(>F)`[3] < 0.05) {
  emm_8 <- emmeans(model_8, pairwise ~ Genotype | ISI_Time)
  contrasts_8 <- summary(emm_8$contrasts, adjust = "none")
  print(contrasts_8)
}

Type III Analysis of Variance Table with Satterthwaite's method
                   Sum Sq Mean Sq NumDF DenDF F value   Pr(>F)    
Genotype            59.54   59.54     1    19  4.7110  0.04285 *  
ISI_Time          1735.07  433.77     4    76 34.3203 2.38e-16 ***
Genotype:ISI_Time  125.19   31.30     4    76  2.4764  0.05117 .  
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [24]:
model_9 <- lmer(
  EPSP_Amplitude ~ Drug + Drug * ISI_Time + (1 | Subject),
  data = sc_gnb1_data,
  REML = TRUE
)

print(anova(model_9))

if (anova(model_9)$`Pr(>F)`[3] < 0.05) {
  emm_9 <- emmeans(model_9, pairwise ~ Drug | ISI_Time)
  contrasts_9 <- summary(emm_9$contrasts, adjust = "none")
  print(contrasts_9)
}

Type III Analysis of Variance Table with Satterthwaite's method
              Sum Sq Mean Sq NumDF  DenDF F value    Pr(>F)    
Drug          212.84 212.843     1 102.91 67.7399 6.127e-13 ***
ISI_Time      998.33 249.582     4 101.00 79.4326 < 2.2e-16 ***
Drug:ISI_Time  64.61  16.152     4 101.00  5.1405 0.0008234 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1
ISI_Time = ISI10:
 contrast      estimate    SE  df t.ratio p.value
 Drug0 - Drug1    -5.06 0.737 101  -6.857  <.0001

ISI_Time = ISI25:
 contrast      estimate    SE  df t.ratio p.value
 Drug0 - Drug1    -4.02 0.737 101  -5.449  <.0001

ISI_Time = ISI50:
 contrast      estimate    SE  df t.ratio p.value
 Drug0 - Drug1    -2.70 0.737 101  -3.667  0.0004

ISI_Time = ISI100:
 contrast      estimate    SE  df t.ratio p.value
 Drug0 - Drug1    -1.64 0.737 101  -2.223  0.0284

ISI_Time = ISI300:
 contrast      estimate    SE  df t.ratio p.value
 Drug0 - Drug1    -1.10 0.737 101  -1.491  0.1390

Degrees-of-freed

In [25]:
model_10 <- lmer(
  EPSP_Amplitude ~ Drug + Drug * ISI_Time + (1 | Subject),
  data = sc_gnb1_data,
  REML = TRUE
)

print(anova(model_10))  

if (anova(model_10)$`Pr(>F)`[3] < 0.05) {
  emm_10 <- emmeans(model_10, pairwise ~ Drug | ISI_Time)
  contrasts_10 <- summary(emm_10$contrasts, adjust = "none")
  print(contrasts_10)
}


Type III Analysis of Variance Table with Satterthwaite's method
              Sum Sq Mean Sq NumDF  DenDF F value    Pr(>F)    
Drug          212.84 212.843     1 102.91 67.7399 6.127e-13 ***
ISI_Time      998.33 249.582     4 101.00 79.4326 < 2.2e-16 ***
Drug:ISI_Time  64.61  16.152     4 101.00  5.1405 0.0008234 ***
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1
ISI_Time = ISI10:
 contrast      estimate    SE  df t.ratio p.value
 Drug0 - Drug1    -5.06 0.737 101  -6.857  <.0001

ISI_Time = ISI25:
 contrast      estimate    SE  df t.ratio p.value
 Drug0 - Drug1    -4.02 0.737 101  -5.449  <.0001

ISI_Time = ISI50:
 contrast      estimate    SE  df t.ratio p.value
 Drug0 - Drug1    -2.70 0.737 101  -3.667  0.0004

ISI_Time = ISI100:
 contrast      estimate    SE  df t.ratio p.value
 Drug0 - Drug1    -1.64 0.737 101  -2.223  0.0284

ISI_Time = ISI300:
 contrast      estimate    SE  df t.ratio p.value
 Drug0 - Drug1    -1.10 0.737 101  -1.491  0.1390

Degrees-of-freed

# Supralinearity + E:I imbalance

In [26]:
file_name <- '/Users/samgritz/Desktop/Rutgers/Milstein_Lab/Code/Rutgers-Neuroscience-PhD/GNB1_Paper_Analysis/paper_data/E_I_data/E_I_amplitudes.csv'

if (!file.exists(file_name)) {
  stop("ERROR: Data file not found: ", file_name, "\n",
       "Please run Analyze_and_Export_E_I_data.py first to generate data.")
}

E_I_data <- read.csv(file_name)

E_I_clean <- E_I_data %>%
  mutate(
    Subject = as.factor(Cell_ID),
    Genotype = as.factor(Genotype),
    Pathway = as.factor(Pathway),
    ISI_Time = as.factor(paste0("ISI", ISI))
  )
E_I_clean.keys <- names(E_I_clean)
print(E_I_clean.keys)

 [1] "Cell_ID"                        "Genotype"                      
 [3] "Sex"                            "ISI"                           
 [5] "Channel"                        "Pathway"                       
 [7] "Control_Amplitude"              "Gabazine_Amplitude"            
 [9] "Gabazine...Etx_Amplitude"       "Estimated_Inhibition_Amplitude"
[11] "E_I_Imbalance"                  "Expected_EPSP_Amplitude"       
[13] "Control_Supralinearity"         "Gabazine_Supralinearity"       
[15] "Gabazine...Ml297_Amplitude"     "Unknown_Amplitude"             
[17] "Gabazine...Mcpg_Amplitude"      "Gabazine...Baclofen_Amplitude" 
[19] "Subject"                        "ISI_Time"                      


In [45]:
#formula String for Gabazine Supralinearity
formula_str_supra <- paste('Gabazine_Supralinearity', "~ Genotype + Genotype * ISI_Time + (1 | Subject)")

#formula String for E_I_Imbalance
formula_str_e_i <- paste('E_I_Imbalance', "~ Genotype + Genotype * ISI_Time + (1 | Subject)")


### Gabazine Supralinearity

In [46]:
# Filter for just Gabazine Supralinearity data (Perforant pathway)
gabazine_supralinearity_pp <- E_I_clean %>% 
    filter(Pathway == 'Perforant' & !is.na(.data[['Gabazine_Supralinearity']])) %>%
    dplyr::select(Cell_ID, Subject, Genotype, ISI_Time, Gabazine_Supralinearity)

gabazine_pp_supra_model <- tryCatch({
lmer(as.formula(formula_str_supra), data = gabazine_supralinearity_pp, REML = TRUE)
}, error = function(e) {
cat("  Error fitting model:", conditionMessage(e), "\n")
return(NULL)
})
    
print(anova(gabazine_pp_supra_model))

if (anova(gabazine_pp_supra_model )$`Pr(>F)`[3] < 0.05) {
  gabazine_pp_supra <- emmeans(gabazine_pp_supra_model, pairwise ~ Genotype | ISI_Time)
  gabazine_pp_supra_contrasts <- summary(gabazine_pp_supra$contrasts, adjust = "none")
  print(gabazine_pp_supra_contrasts)
}

Type III Analysis of Variance Table with Satterthwaite's method
                  Sum Sq Mean Sq NumDF   DenDF F value    Pr(>F)    
Genotype          111.36 111.364     1  87.197  9.4012  0.002887 ** 
ISI_Time          547.73 136.932     4 145.992 11.5596 3.516e-08 ***
Genotype:ISI_Time 222.82  55.704     4 145.992  4.7024  0.001343 ** 
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1
ISI_Time = ISI10:
 contrast  estimate    SE  df t.ratio p.value
 GNB1 - WT    6.948 1.590 172   4.374  <.0001

ISI_Time = ISI100:
 contrast  estimate    SE  df t.ratio p.value
 GNB1 - WT    0.569 1.460 170   0.389  0.6981

ISI_Time = ISI25:
 contrast  estimate    SE  df t.ratio p.value
 GNB1 - WT    3.311 1.570 172   2.112  0.0361

ISI_Time = ISI300:
 contrast  estimate    SE  df t.ratio p.value
 GNB1 - WT    0.000 0.935 145   0.000  1.0000

ISI_Time = ISI50:
 contrast  estimate    SE  df t.ratio p.value
 GNB1 - WT    2.373 1.530 171   1.548  0.1234

Degrees-of-freedom method: kenward-r

In [47]:
gabazine_supralinearity_sc <- E_I_clean %>% 
    filter(Pathway == 'Schaffer' & !is.na(.data[['Gabazine_Supralinearity']])) %>%
    dplyr::select(Cell_ID, Subject, Genotype, ISI_Time, Gabazine_Supralinearity)

gabazine_sc_supra_model <- tryCatch({
lmer(as.formula(formula_str_supra), data = gabazine_supralinearity_sc, REML = TRUE)
}, error = function(e) {
cat("  Error fitting model:", conditionMessage(e), "\n")
return(NULL)
})
    
print(anova(gabazine_sc_supra_model))

if (anova(gabazine_sc_supra_model )$`Pr(>F)`[3] < 0.05) {
  gabazine_sc_supra <- emmeans(gabazine_sc_supra_model, pairwise ~ Genotype | ISI_Time)
  gabazine_sc_supra_contrasts <- summary(gabazine_sc_supra$contrasts, adjust = "none")
  print(gabazine_sc_supra_contrasts)
}

Type III Analysis of Variance Table with Satterthwaite's method
                  Sum Sq Mean Sq NumDF   DenDF F value    Pr(>F)    
Genotype            3.25   3.251     1  82.296  0.4577    0.5006    
ISI_Time          395.23  98.807     4 139.420 13.9108 1.421e-09 ***
Genotype:ISI_Time   9.41   2.353     4 139.420  0.3312    0.8566    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [48]:
gabazine_supralinearity_so <- E_I_clean %>% 
    filter(Pathway == 'Basal_Stratum_Oriens' & !is.na(.data[['Gabazine_Supralinearity']])) %>%
    dplyr::select(Cell_ID, Subject, Genotype, ISI_Time, Gabazine_Supralinearity)

gabazine_so_supra_model <- tryCatch({
lmer(as.formula(formula_str_supra), data = gabazine_supralinearity_so, REML = TRUE)
}, error = function(e) {
cat("  Error fitting model:", conditionMessage(e), "\n")
return(NULL)
})
    
print(anova(gabazine_so_supra_model))

if (anova(gabazine_so_supra_model )$`Pr(>F)`[3] < 0.05) {
  gabazine_so_supra <- emmeans(gabazine_so_supra_model, pairwise ~ Genotype | ISI_Time)
  gabazine_so_supra_contrasts <- summary(gabazine_so_supra$contrasts, adjust = "none")
  print(gabazine_so_supra_contrasts)
}

Type III Analysis of Variance Table with Satterthwaite's method
                  Sum Sq Mean Sq NumDF  DenDF F value    Pr(>F)    
Genotype            8.04   8.035     1 19.816  0.7193    0.4065    
ISI_Time          330.98  82.746     4 78.949  7.4070 4.054e-05 ***
Genotype:ISI_Time  53.59  13.399     4 78.949  1.1994    0.3177    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


### E_I_Imbalance

In [ ]:
e_i_imbalance_pp <- E_I_clean %>% 
    filter(Pathway == 'Perforant' & !is.na(.data[['E_I_Imbalance']])) %>%
    dplyr::select(Cell_ID, Subject, Genotype, ISI_Time, E_I_Imbalance)

e_i_imbalance_pp_model <- tryCatch({
lmer(as.formula(formula_str_e_i), data = e_i_imbalance_pp, REML = TRUE)
}, error = function(e) {
cat("  Error fitting model:", conditionMessage(e), "\n")
return(NULL)
})
    
print(anova(e_i_imbalance_pp_model))

if (anova(e_i_imbalance_pp_model)$`Pr(>F)`[3] < 0.05) {
  e_i_imbalance_pp_em <- emmeans(e_i_imbalance_pp_model, pairwise ~ Genotype | ISI_Time)
  e_i_imbalance_pp_post_hoc <- summary(e_i_imbalance_pp_em$contrasts, adjust = "none")
  print(e_i_imbalance_pp_post_hoc)
}

Type III Analysis of Variance Table with Satterthwaite's method
                    Sum Sq  Mean Sq NumDF   DenDF F value    Pr(>F)    
Genotype          0.000612 0.000612     1  80.931  0.1332    0.7161    
ISI_Time          0.177018 0.044254     4 108.045  9.6251 1.074e-06 ***
Genotype:ISI_Time 0.002091 0.000523     4 108.045  0.1137    0.9775    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [ ]:
e_i_imbalance_sc <- E_I_clean %>% 
    filter(Pathway == 'Schaffer' & !is.na(.data[['E_I_Imbalance']])) %>%
    dplyr::select(Cell_ID, Subject, Genotype, ISI_Time, E_I_Imbalance)

e_i_imbalance_sc_model <- tryCatch({
lmer(as.formula(formula_str_e_i), data = e_i_imbalance_sc, REML = TRUE)
}, error = function(e) {
cat("  Error fitting model:", conditionMessage(e), "\n")
return(NULL)
})
    
print(anova(e_i_imbalance_sc_model))

if (anova(e_i_imbalance_sc_model)$`Pr(>F)`[3] < 0.05) {
  e_i_imbalance_sc_em <- emmeans(e_i_imbalance_sc_model, pairwise ~ Genotype | ISI_Time)
  e_i_imbalance_sc_post_hoc <- summary(e_i_imbalance_sc_em$contrasts, adjust = "none")
  print(e_i_imbalance_sc_post_hoc)
}

Type III Analysis of Variance Table with Satterthwaite's method
                    Sum Sq   Mean Sq NumDF  DenDF F value    Pr(>F)    
Genotype          0.000948 0.0009479     1 73.082  0.2428 0.6236578    
ISI_Time          0.078558 0.0196396     4 98.040  5.0311 0.0009909 ***
Genotype:ISI_Time 0.007433 0.0018581     4 98.040  0.4760 0.7532529    
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1


In [56]:
e_i_imbalance_so <- E_I_clean %>% 
    filter(Pathway == 'Basal_Stratum_Oriens' & !is.na(.data[['E_I_Imbalance']])) %>%
    dplyr::select(Cell_ID, Subject, Genotype, ISI_Time, E_I_Imbalance)

e_i_imbalance_so_model <- tryCatch({
lmer(as.formula(formula_str_e_i), data = e_i_imbalance_so, REML = TRUE)
}, error = function(e) {
cat("  Error fitting model:", conditionMessage(e), "\n")
return(NULL)
})
    
print(anova(e_i_imbalance_so_model))

if (anova(e_i_imbalance_so_model)$`Pr(>F)`[3] < 0.05) {
  e_i_imbalance_so_em <- emmeans(e_i_imbalance_so_model, pairwise ~ Genotype | ISI_Time)
  e_i_imbalance_so_post_hoc <- summary(e_i_imbalance_so_em$contrasts, adjust = "none")
  print(e_i_imbalance_so_post_hoc)
}

Type III Analysis of Variance Table with Satterthwaite's method
                    Sum Sq  Mean Sq NumDF  DenDF F value   Pr(>F)   
Genotype          0.041550 0.041550     1 20.053  6.5505 0.018671 * 
ISI_Time          0.103936 0.025984     4 79.122  4.0965 0.004545 **
Genotype:ISI_Time 0.023103 0.005776     4 79.122  0.9106 0.462015   
---
Signif. codes:  0 ‘***’ 0.001 ‘**’ 0.01 ‘*’ 0.05 ‘.’ 0.1 ‘ ’ 1
